In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
import pandas as pd

-> Chamados
    ->  Filas
      ->   Setores


In [ ]:
chamados = pd.read_csv("/MBA Esalq/TCC/EXPERIMENTOS/chamados_base_de_dados/chamados.csv")
chamados.head()

In [ ]:
print(chamados.shape)

1) excluir as colunas "nro", "status", "fechado_em", "created_at" e "updated_at"
2) verificar coluna "extras" - PODE EXCLUIR TAMBÉM

In [ ]:
chamados_limpo = chamados.drop(columns=["nro", "status", "fechado_em", "extras", "created_at", "updated_at"])
chamados_limpo.head()

In [ ]:
print(chamados_limpo.shape)

In [ ]:
filas = pd.read_csv("/MBA Esalq/TCC/EXPERIMENTOS/chamados_base_de_dados/filas.csv")
filas.head()

In [ ]:
print(filas.shape)

In [ ]:
quant_filas_em_producao = filas[filas["estado"] == "Em produção"].shape[0]
print(f"Quantidade de filas em produção: {quant_filas_em_producao}")
print(f"Quantidade de filas desativadas: {filas[filas["estado"] == "Desativada"].shape[0]}")

In [ ]:
filas_para_merge = filas[['id', 'nome']]
# Ao mesclar, o 'id' de chamados_limpo permanece 'id' e o 'id' de filas_para_merge se torna 'id_y'.
chamados_com_filas = pd.merge(chamados_limpo, filas_para_merge, left_on='fila_id', right_on='id', how='left')
# Agora, descartamos 'fila_id' e 'id_y' (o id da tabela 'filas'), mas mantemos o 'id' original dos chamados.
chamados_com_filas = chamados_com_filas.drop(columns=['fila_id', 'id_y']).rename(columns={'nome': 'fila_nome'})
display(chamados_com_filas.head())

In [ ]:
comentarios = pd.read_csv("/MBA Esalq/TCC/EXPERIMENTOS/chamados_base_de_dados/comentarios.csv")
comentarios.head()

1) selecionar apenas as linhas com o "tipo" = user
2) excluir as colunas "id", "user_id", "chamado_id", "created_at" e "updated_at"
3) agrupar "comentario" por "chamado_id"


In [ ]:
comentarios_limpo = comentarios[comentarios["tipo"] == "user"]
comentarios_limpo = comentarios_limpo.drop(columns=["id", "user_id", "created_at", "updated_at"])
comentarios_limpo.head()

In [ ]:
comentarios_agrupados = comentarios_limpo.groupby('chamado_id')['comentario'].apply(lambda x: '. '.join(x)).reset_index()
display(comentarios_agrupados.head())

In [ ]:
chamados_com_filas = pd.merge(chamados_com_filas, comentarios_agrupados, left_on='id_x', right_on='chamado_id', how='left')
chamados_com_filas = chamados_com_filas.drop(columns=['chamado_id']) # Drop the redundant 'chamado_id' column after merge
display(chamados_com_filas.head())

In [ ]:
print(chamados_com_filas.shape)

In [ ]:
chamados_com_filas['anotacoes'] = chamados_com_filas['anotacoes'].fillna('anotação não informada')
display(chamados_com_filas.head())

In [ ]:
num_anotacoes_informadas = chamados_com_filas[chamados_com_filas['anotacoes'] != 'anotação não informada'].shape[0]
print(f"Número de registros com anotações diferentes de 'anotação não informada': {num_anotacoes_informadas}")

In [ ]:
num_anotacoes_nao_informadas = chamados_com_filas[chamados_com_filas['anotacoes'] == 'anotação não informada'].shape[0]
print(f"Número de registros com 'anotação não informada': {num_anotacoes_nao_informadas}")

Separar em arquivos por nome da fila: 31 filas total - 28 em produção, 3 desativadas.

Ao importar para o banco vetorial, será adicionado um novo metadado informando a origem do texto/informação.

## Tarefa
Criado arquivos CSV separados por 'fila_nome' a partir do dataset 'chamados_com_filas'.

In [ ]:
unique_filas = chamados_com_filas['fila_nome'].unique()
print(f"Unique Fila Names: {unique_filas}")
print(f"Number of unique filas: {len(unique_filas)}")

### Subtarefa:
Criado um diretório para armazenar os arquivos CSV gerados, caso esse diretório ainda não exista.

#### Instruções:
1. Import the `os` module.
2. Define the path for the output directory.
3. Use `os.makedirs()` with `exist_ok=True` to create the directory.

In [ ]:
import os

output_directory = "/MBA Esalq/TCC/EXPERIMENTOS/chamados_por_fila/"
os.makedirs(output_directory, exist_ok=True)

print(f"Output directory '{output_directory}' ensured to exist.")

### Subtarefa:

Percorre cada `fila_nome` único e salva um arquivo CSV separado para cada uma. Cada arquivo CSV deve conter apenas as linhas correspondentes àquele `fila_nome`.

#### Instruções:
1. Percorre cada `fila_nome` no array `unique_filas`.
2. Para cada `fila_nome`, filtra o dataset `chamados_com_filas` para obter apenas as linhas onde a coluna `fila_nome` corresponde ao `fila_nome` atual.
3. Cria um nome de arquivo válido para o CSV, substituindo quaisquer caracteres problemáticos em `fila_nome` (por exemplo, espaços, barras) por sublinhados ou removendo-os e adiciona `.csv` ao nome.
4. Salva o dataset filtrado em um arquivo CSV no diretório `output_directory` usando `index=False` para evitar a gravação do índice do DataFrame no CSV.

In [ ]:
import re

for fila_nome in unique_filas:
    # Filtra o dataset para o 'fila_nome' atual
    df_filtered = chamados_com_filas[chamados_com_filas['fila_nome'] == fila_nome]

    # Limpa o 'fila_nome' para criar um nome de arquivo válido
    # Substitui os caracteres não alfanuméricos (exceto espaços) por sublinhados
    sanitized_fila_name = re.sub(r'[^a-zA-Z0-9_ -]', '', fila_nome)
    # Substitui os espaços por sublinhados e remover espaços em branco iniciais e finais
    sanitized_fila_name = sanitized_fila_name.replace(' ', '_').strip('_')

    # Constroi o caminho completo para o arquivo TXT
    file_path = os.path.join(output_directory, f"{sanitized_fila_name}.txt")

    # Salva o dataset filtrado em um arquivo TXT, usando tabulação como separador
    df_filtered.to_csv(file_path, index=False, sep='\t')
    print(f"Saved {len(df_filtered)} records for '{fila_nome}' to {file_path}")

## Gerando Arquivos TXT para Cada Registro

Para o experimento onde cada registro é um chunk.

### Subtarefa:
Cria um arquivo TXT separado para cada registro individual (linha) no dataset `chamados_com_filas`. Cada arquivo deve conter todos os dados das colunas para aquele registro específico.

#### Instruções:
1. Defina um novo diretório de saída, `/MBA Esalq/TCC/chamados_por_registro/`.
2. Certifique-se de que este diretório exista.
3. Itere por cada linha do dataset `chamados_com_filas`.
4. Para cada linha, construa um nome de arquivo único usando o `id` e uma versão anonimizada da coluna `assunto`.
5. Escreva os dados de toda a linha no arquivo TXT, com cada coluna apresentada como 'Nome da Coluna: Valor'.

In [ ]:
import os
import re

# Define o novo diretório para arquivos de registro individuais
output_directory_per_record = "/MBA Esalq/TCC/EXPERIMENTOS/chamados_por_registro/"

# Cria o diretório se ele não existir
os.makedirs(output_directory_per_record, exist_ok=True)
print(f"Output directory '{output_directory_per_record}' ensured to exist.")

In [ ]:
# Itera por cada linha do dataset chamados_com_filas
for index, row in chamados_com_filas.iterrows():
    # Obtem o ID e o assunto do nome do arquivo
    record_id = row['id_x']
    assunto = row['assunto']

    # Limpar o 'assunto' para criar uma parte válida do nome do arquivo
    # Remove os caracteres não alfanuméricos e substitui os espaços por sublinhados
    sanitized_assunto = re.sub(r'[^a-zA-Z0-9_ -]', '', str(assunto))
    sanitized_assunto = sanitized_assunto.replace(' ', '_').strip('_')
    
    # Constroi o nome do arquivo: por exemplo, "id_assunto.txt"
    file_name = f"{record_id}_{sanitized_assunto}.txt"
    file_path = os.path.join(output_directory_per_record, file_name)

    # Prepara o conteúdo para o arquivo TXT
    # Escreve cada coluna como 'Nome da Coluna: Valor'
    content = []
    for col_name, value in row.items():
        content.append(f"{col_name}: {value}")

    # Escreve o conteúdo no arquivo TXT
    with open(file_path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(content))

    print(f"Saved record ID {record_id} to {file_path}")

print(f"\nFinished saving {len(chamados_com_filas)} records to individual TXT files.")


In [ ]:
import os
import re

# Define o novo diretório para os arquivos de registro individuais (presumindo que já tenha sido criado nas etapas anteriores)
output_directory_per_record = "/MBA Esalq/TCC/EXPERIMENTOS/chamados_por_registro/fila_hospedagens_incidentes/"

# Filtra o dataset para o 'fila_nome' específico
sistemas_formularios_df = chamados_com_filas[chamados_com_filas['fila_nome'] == 'Hospedagens - Incidentes de vulnerabilidade']

# Cria o diretório se ele não existir
os.makedirs(output_directory_per_record, exist_ok=True)
print(f"Output directory '{output_directory_per_record}' ensured to exist.")

In [ ]:
# Percorre cada linha do dataset filtrado
for index, row in sistemas_formularios_df.iterrows():
    # Get the ID and assunto for the filename
    record_id = row['id_x']
    assunto = row['assunto']

    # Limpa o 'assunto' para criar uma parte válida do nome do arquivo
    # Remove os caracteres não alfanuméricos e substitui os espaços por sublinhados
    sanitized_assunto = re.sub(r'[^a-zA-Z0-9_ -]', '', str(assunto))
    sanitized_assunto = sanitized_assunto.replace(' ', '_').strip('_')

    # Constroi o nome do arquivo: por exemplo, "id_assunto.txt"
    file_name = f"{record_id}_{sanitized_assunto}.txt"
    file_path = os.path.join(output_directory_per_record, file_name)

    # Prepara o conteúdo para o arquivo TXT
    # Escreve cada coluna como 'Nome da Coluna: Valor'
    content = []
    for col_name, value in row.items():
        content.append(f"{col_name}: {value}")

    # Escreva o conteúdo no arquivo TXT
    with open(file_path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(content))

    print(f"Saved record ID {record_id} (Sistemas de formulários) to {file_path}")

print(f"\nFinished saving {len(sistemas_formularios_df)} records from 'Sistemas de formulários' to individual TXT files.")